# Startech.com.bd — Monitor Category Scraper + Bulk-Upload Mapper

This notebook does three things, run top to bottom:

1. **Part 1 — Category listing scrape**: walks every page of
   https://www.startech.com.bd/monitor and collects the summary card for
   every product (name, price, brand, short specs, image, url, id).
2. **Part 2 — Product detail scrape**: visits each product's own page and
   pulls the full specification table, description, ratings, gallery
   images and price breakdown, then merges it onto the listing data.
3. **Part 3 — Bulk-upload mapping**: reads `sample_template.xlsx` (the
   store's bulk-import format — sheets `Products`, `Categories`, `Brands`,
   `Instructions`) and maps the scraped data onto its `Products` columns,
   producing one ready-to-import bulk-upload workbook per brand under
   `bulk_upload/<category>/<brand>/`.

> **Before you import that file, read the callouts in Part 3** — a couple
> of fields (Current Cost, Description, SKU) need a conscious decision
> from you, they are not something this script can respect unattended.

Change `CATEGORY_URL` in the config cell below to point this whole
pipeline at a different Startech category (e.g. `.../laptop`) — every
other cell is category-agnostic. That same cell also has `BRAND_FILTER`
(only scrape/map one brand) and `PRODUCT_LIMIT` (cap the total product
count, e.g. `5` for a quick end-to-end test run) — both optional, both
`None` by default meaning "everything".

In [ ]:
import re
import csv
import json
import time
import hashlib
import threading
from pathlib import Path
from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from bs4 import BeautifulSoup

# ---- Config -----------------------------------------------------------
CATEGORY_URL = "https://www.startech.com.bd/ups"
CATEGORY_NAME = "ups"          # used to tag records + name output files
OUT_DIR = Path(".")   # this notebook lives inside startech_scrape/, alongside the template + data files
OUT_DIR.mkdir(exist_ok=True)

LIMIT = 90                          # largest page size the site's dropdown offers (unrelated to PRODUCT_LIMIT below)
DETAIL_WORKERS = 8                  # concurrent threads for product-detail fetches
REQUEST_TIMEOUT = 25
MAX_RETRIES = 3

BRAND_FILTER = None                 # e.g. "Samsung" -> only keep products matching that brand (case-insensitive,
                                     # checked against both the brand field and the product name so multi-word
                                     # brands like "PC Power" still match). None/"" = every brand.
PRODUCT_LIMIT = None                 # e.g. 5 -> only keep the first N products after any brand filter.
                                     # None = no cap (all matching products). Handy for a quick dry run of the
                                     # whole pipeline without waiting to scrape hundreds of detail pages.

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}


def filter_products(products, brand_filter=BRAND_FILTER, limit=PRODUCT_LIMIT):
    """Apply BRAND_FILTER then PRODUCT_LIMIT to a list of product dicts. Shared
    by Part 1 (right after the listing scrape, so Part 2 doesn't waste requests
    fetching detail pages that would just get dropped) and Part 3 (so re-running
    just Part 3 against an already-scraped full.json with a new BRAND_FILTER/
    PRODUCT_LIMIT re-slices it without re-scraping)."""
    result = products
    if brand_filter:
        needle = brand_filter.strip().lower()
        result = [
            p for p in result
            if needle in (p.get("detail_brand") or p.get("brand") or "").lower()
            or needle in (p.get("detail_name") or p.get("name") or "").lower()
        ]
    if limit:
        result = result[:limit]
    return result

## Part 1 — Category listing scrape

Fetches `CATEGORY_URL?limit=90&page=N` until the page count from the
"Showing X to Y of Z (N Pages)" banner is exhausted, parsing every
`div.p-item` card.

In [2]:
def clean_price(text):
    """'4,300৳' -> 4300 (int); None if not parseable."""
    if not text:
        return None
    digits = re.sub(r"[^\d]", "", text)
    return int(digits) if digits else None


def parse_listing_products(html):
    soup = BeautifulSoup(html, "html.parser")
    products = []

    for item in soup.select("div.p-item"):
        name_tag = item.select_one("h4.p-item-name a")
        name = name_tag.get_text(strip=True) if name_tag else None
        url = name_tag["href"] if name_tag and name_tag.has_attr("href") else None
        if url:
            url = url.strip()  # a few source hrefs carry a trailing space -> bad %20 URLs

        img_tag = item.select_one(".p-item-img img")
        image = None
        if img_tag:
            image = img_tag.get("data-src") or img_tag.get("src")
            if image:
                image = urljoin(CATEGORY_URL, image)

        price_new_tag = item.select_one(".p-item-price .price-new")
        price_old_tag = item.select_one(".p-item-price .price-old")
        if price_new_tag:
            price = clean_price(price_new_tag.get_text())
            regular_price = clean_price(price_old_tag.get_text()) if price_old_tag else price
        else:
            price_span = item.select_one(".p-item-price span")
            price = clean_price(price_span.get_text()) if price_span else None
            regular_price = price

        save_tag = item.select_one(".marks .mark")
        save_amount = clean_price(save_tag.get_text()) if save_tag else None

        specs = []
        desc_tag = item.select_one(".short-description")
        if desc_tag:
            specs = [li.get_text(" ", strip=True) for li in desc_tag.select("li") if li.get_text(strip=True)]

        product_id = None
        add_cart_tag = item.select_one(".btn-add-cart")
        if add_cart_tag and add_cart_tag.has_attr("onclick"):
            m = re.search(r"cart\.add\('(\d+)'", add_cart_tag["onclick"])
            if m:
                product_id = m.group(1)

        stock_tag = item.select_one(".stock-status, .p-item-stock, .out-of-stock")
        stock_status = stock_tag.get_text(strip=True) if stock_tag else "In Stock"

        brand = name.split()[0] if name else None  # rough guess; Part 2's detail_brand is canonical

        products.append({
            "product_id": product_id, "name": name, "brand": brand, "url": url,
            "image": image, "price": price, "regular_price": regular_price,
            "save_amount": save_amount, "stock_status": stock_status, "specs": specs,
        })

    return products


def get_total_pages(html):
    text = BeautifulSoup(html, "html.parser").get_text()
    m = re.search(r"Showing\s+[\d,]+\s+to\s+[\d,]+\s+of\s+([\d,]+)\s*\((\d+)\s*Pages?\)", text)
    if m:
        return int(m.group(1).replace(",", "")), int(m.group(2))
    return None, 1


def scrape_listing(category_url=CATEGORY_URL, limit=LIMIT, delay=1.0, category_name=CATEGORY_NAME):
    session = requests.Session()
    session.headers.update(HEADERS)

    all_products = []
    page, total_pages = 1, None

    while True:
        resp = session.get(category_url, params={"limit": limit, "page": page}, timeout=REQUEST_TIMEOUT)
        resp.raise_for_status()
        html = resp.text

        if total_pages is None:
            total_items, total_pages = get_total_pages(html)
            print(f"[{category_name}] total items: {total_items}, total pages: {total_pages}")

        page_products = parse_listing_products(html)
        if not page_products:
            break
        for p in page_products:
            p["category"] = category_name
        all_products.extend(page_products)
        print(f"[{category_name}] page {page}: +{len(page_products)} (running total {len(all_products)})")

        if total_pages and page >= total_pages:
            break
        page += 1
        time.sleep(delay)

    return all_products

In [3]:
listing = scrape_listing()
print(f"\nScraped {len(listing)} product cards from the listing pages.")

listing = filter_products(listing)
if BRAND_FILTER or PRODUCT_LIMIT:
    print(f"Filtered to {len(listing)} product(s) -- BRAND_FILTER={BRAND_FILTER!r}, PRODUCT_LIMIT={PRODUCT_LIMIT!r}.")

[processor] total items: 121, total pages: 2
[processor] page 1: +90 (running total 90)
[processor] page 2: +31 (running total 121)

Scraped 121 product cards from the listing pages.


In [4]:
listing_json_path = OUT_DIR / f"startech_{CATEGORY_NAME}_listing.json"
listing_csv_path = OUT_DIR / f"startech_{CATEGORY_NAME}_listing.csv"

with open(listing_json_path, "w", encoding="utf-8") as f:
    json.dump(listing, f, ensure_ascii=False, indent=2)

fieldnames = ["product_id", "category", "brand", "name", "price", "regular_price",
              "save_amount", "stock_status", "url", "image", "specs"]
with open(listing_csv_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for p in listing:
        row = dict(p)
        row["specs"] = " | ".join(p.get("specs", []))
        writer.writerow(row)

print("Saved:", listing_json_path, listing_csv_path)

Saved: startech_processor_listing.json startech_processor_listing.csv


## Part 2 — Product detail scrape

Visits every `url` from the listing and pulls: canonical SKU/brand/status,
the full price breakdown (cash price / struck-through price / EMI
reference price), the "Key Features" bullets, the **entire specification
table** grouped by its section headings (Display Features, Connectivity,
Warranty Information, …), the full description text, rating/review
counts, the gallery image URLs, and the EMI-per-month figure.

Runs with a small thread pool since one request per product would take
too long serially; each thread gets its own `requests.Session`.

In [5]:
_thread_local = threading.local()

def get_session():
    if not hasattr(_thread_local, "session"):
        s = requests.Session()
        s.headers.update(HEADERS)
        _thread_local.session = s
    return _thread_local.session


def parse_detail(html, url):
    soup = BeautifulSoup(html, "html.parser")

    sku_meta = soup.select_one('meta[itemprop="sku"]')
    sku = sku_meta["content"] if sku_meta and sku_meta.has_attr("content") else None

    name_tag = soup.select_one("h1.product-name")
    name = name_tag.get_text(strip=True) if name_tag else None

    price = price_old = regular_price = status = brand = None

    price_cell = soup.select_one("td.product-price")
    if price_cell:
        ins, del_ = price_cell.select_one("ins"), price_cell.select_one("del")
        if ins:
            price = clean_price(ins.get_text())
            price_old = clean_price(del_.get_text()) if del_ else None
        else:
            price = clean_price(price_cell.get_text())

    reg_cell = soup.select_one("td.product-regular-price")
    if reg_cell:
        regular_price = clean_price(reg_cell.get_text())

    status_cell = soup.select_one("td.product-status")
    if status_cell:
        status = status_cell.get_text(strip=True)

    code_cell = soup.select_one("td.product-code")
    if code_cell and not sku:
        sku = code_cell.get_text(strip=True)

    brand_cell = soup.select_one("td.product-brand")
    if brand_cell:
        brand = brand_cell.get_text(strip=True)

    key_features = []
    for li in soup.select(".short-description ul li"):
        if "view-more" in (li.get("class") or []):
            continue
        text = li.get_text(" ", strip=True)
        if text:
            key_features.append(text)

    specification = {}
    spec_table = soup.select_one("table.data-table")
    if spec_table:
        current_category = "Other"
        for sec in spec_table.find_all(["thead", "tbody"]):
            if sec.name == "thead":
                heading = sec.select_one("td.heading-row")
                current_category = heading.get_text(strip=True) if heading else "Other"
                specification.setdefault(current_category, {})
            else:
                for tr in sec.find_all("tr"):
                    tds = tr.find_all("td")
                    if len(tds) >= 2:
                        field_name = tds[0].get_text(strip=True)
                        value_td = tds[1]
                        for br in value_td.find_all("br"):
                            br.replace_with(" / ")
                        specification.setdefault(current_category, {})[field_name] = value_td.get_text(strip=True)

    desc_div = soup.select_one(".full-description")
    description = desc_div.get_text("\n", strip=True) if desc_div else None

    rating_value = review_count = None
    rating_meta = soup.select_one('[itemprop="aggregateRating"]')
    if rating_meta:
        rv = rating_meta.select_one('meta[itemprop="ratingValue"]')
        rc = rating_meta.select_one('meta[itemprop="reviewCount"]')
        rating_value = rv["content"] if rv and rv.has_attr("content") else None
        review_count = rc["content"] if rc and rc.has_attr("content") else None

    gallery = []
    for a in soup.select(".product-images a.thumbnail"):
        href = a.get("href")
        if href and href not in gallery:
            gallery.append(href)

    emi_price_per_month = None
    emi_tag = soup.select_one(".p-wrap.emi .price")
    if emi_tag:
        emi_price_per_month = emi_tag.get_text(strip=True)

    return {
        "sku": sku, "detail_name": name, "detail_price": price, "detail_price_old": price_old,
        "detail_regular_price": regular_price, "status": status, "detail_brand": brand,
        "key_features": key_features, "specification": specification, "description": description,
        "rating_value": rating_value, "review_count": review_count, "gallery_images": gallery,
        "emi_price_per_month": emi_price_per_month, "detail_url": url,
    }


def fetch_one(url):
    session = get_session()
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = session.get(url, timeout=REQUEST_TIMEOUT)
            resp.raise_for_status()
            return parse_detail(resp.text, url)
        except Exception as e:
            last_err = e
            time.sleep(0.5 * attempt)
    return {"detail_url": url, "error": str(last_err)}

In [6]:
urls = [p["url"] for p in listing if p.get("url")]
print(f"Fetching detail pages for {len(urls)} products with {DETAIL_WORKERS} workers...")

results_by_url = {}
done = errors = 0

with ThreadPoolExecutor(max_workers=DETAIL_WORKERS) as executor:
    future_to_url = {executor.submit(fetch_one, u): u for u in urls}
    for future in as_completed(future_to_url):
        url = future_to_url[future]
        detail = future.result()
        if detail.get("error"):
            errors += 1
        results_by_url[url] = detail
        done += 1
        if done % 50 == 0 or done == len(urls):
            print(f"  progress: {done}/{len(urls)} (errors so far: {errors})")

print(f"\nDone. {done - errors}/{done} succeeded, {errors} failed.")

Fetching detail pages for 121 products with 8 workers...
  progress: 50/121 (errors so far: 0)
  progress: 100/121 (errors so far: 0)
  progress: 121/121 (errors so far: 0)

Done. 121/121 succeeded, 0 failed.


In [7]:
# Retry anything that failed once (transient blocks / stray whitespace in the source href)
failed_urls = [u for u, d in results_by_url.items() if d.get("error")]
if failed_urls:
    print(f"Retrying {len(failed_urls)} failed URL(s)...")
    for u in failed_urls:
        clean_url = u.strip()
        detail = fetch_one(clean_url)
        if not detail.get("error"):
            results_by_url[u] = detail
            print("  fixed:", clean_url)
        else:
            print("  still failing:", clean_url, "->", detail["error"])
else:
    print("No failed URLs to retry.")

No failed URLs to retry.


In [8]:
# Merge listing + detail into one record per product
merged = []
for p in listing:
    d = results_by_url.get(p.get("url"), {})
    rec = dict(p)
    rec.update({k: v for k, v in d.items() if k != "detail_url"})
    rec["detail_url"] = d.get("detail_url", p.get("url"))
    merged.append(rec)

full_json_path = OUT_DIR / f"startech_{CATEGORY_NAME}_full.json"
full_csv_path = OUT_DIR / f"startech_{CATEGORY_NAME}_full.csv"

with open(full_json_path, "w", encoding="utf-8") as f:
    json.dump(merged, f, ensure_ascii=False, indent=2)

flat_fields = ["product_id", "sku", "category", "brand", "name", "price", "regular_price",
               "detail_price", "detail_price_old", "detail_regular_price", "status",
               "stock_status", "save_amount", "rating_value", "review_count",
               "emi_price_per_month", "url", "image", "specs", "key_features",
               "specification", "description"]
with open(full_csv_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=flat_fields)
    writer.writeheader()
    for p in merged:
        row = {k: p.get(k) for k in flat_fields}
        row["specs"] = " | ".join(p.get("specs", []) or [])
        row["key_features"] = " | ".join(p.get("key_features", []) or [])
        spec_flat = [f"{cat} > {k}: {v}" for cat, kvs in (p.get("specification") or {}).items() for k, v in kvs.items()]
        row["specification"] = " || ".join(spec_flat)
        row["description"] = (p.get("description") or "").replace("\n", " ")
        writer.writerow(row)

print(f"Saved {len(merged)} merged product records -> {full_json_path}, {full_csv_path}")

Saved 121 merged product records -> startech_processor_full.json, startech_processor_full.csv


## Part 3 — Map onto the bulk-upload template

> **The `.xlsx` export below is currently commented out** (using Part 4's direct-to-admin
> path instead) -- so `sample_template.xlsx` no longer needs to exist for this notebook to
> run; the field-mapping cells here still run (Part 4 needs their output, `rows`), they just
> don't require or produce the spreadsheet anymore. The table below still describes exactly
> what each field means either way.

`sample_template.xlsx` has four sheets: `Products` (the sheet to fill —
one row per product), `Categories` (valid values for the `Category`
column), `Brands` (existing brand names — an unrecognized one is created
automatically per the `Instructions` sheet), and `Instructions` itself.

**Output layout**: with `SPLIT_BY_BRAND = True` (the default, set in the
config cell below), Part 3 writes **one workbook per brand** instead of a
single combined file:

```
bulk_upload/
  monitor/
    samsung/monitor_samsung_bulk_upload.xlsx
    lg/monitor_lg_bulk_upload.xlsx
    dell/monitor_dell_bulk_upload.xlsx
    ...
```

This is what keeps each file's size in check now that every row can carry
several embedded photos (see the caution below) — rather than shrinking the
images or waiting to see if one giant file gets too big, splitting by brand
bounds each file's size by construction. Set `SPLIT_BY_BRAND = False` to get
a single combined file at `BULK_OUT_PATH` instead, same as before. Combine
this with `BRAND_FILTER` (config cell, top of the notebook) to scrape/map
just one brand in the first place — with a single brand selected there's
naturally just one output file.

Field mapping used below, and **why**:

| Template column | Source | Notes |
|---|---|---|
| Name | scraped product name | as-is |
| Name Bengali | *(blank)* | not scraped — Startech's listing is English-only |
| SKU | `generate_sku(...)` → `"<SKU_PREFIX>-<8-char code>"`, e.g. `"GR-A3F9B21C"` | `SKU_PREFIX` defaults to `"GR"`. The code is a *hash* of Startech's product_id (not literally random) so the same product always gets the same SKU on re-runs — true per-run randomness would make the store treat every re-import as a brand-new product. Toggle `INCLUDE_SKU=False` to leave blank and let the store auto-generate instead. |
| Category | `TEMPLATE_CATEGORY` | fixed to `"Monitor"`, verified to match the Categories sheet exactly |
| Brand | `detail_brand` (canonical, from the product page) → falls back to the listing's guessed brand | this is the product's own manufacturer (Samsung, LG, …) — unrelated to `STORE_NAME` below, and left as-scraped since the Categories/Brands sheets validate against real brand names. Also what `SPLIT_BY_BRAND` groups files by. |
| Price / Discount Price | `detail_price_old` / `detail_price` | if there's no struck-through original price, `Price` = current price and `Discount Price` is left blank |
| Stock Quantity / Always In Stock | derived from `status` | "In Stock" → Always In Stock=Yes; "Out Of Stock" → No, qty 0; Pre Order/Up Coming → No, qty blank |
| Current Cost | ***left blank*** | this is the store's internal purchase cost — Startech's retail price is not a substitute, and guessing a margin would fabricate business data. Fill it in yourselves. |
| Unit | `"piece"` | |
| Short Description | first 3 sanitized `key_features` bullets, joined | |
| Description | **generated by `build_description()`**, not scraped verbatim | see below |
| Model | heuristic extraction from the product name | best-effort regex (first alphanumeric token containing a digit); spot-check before trusting it |
| Warranty Value / Unit | parsed from the spec table's "Warranty" text | free text like "03 years of Service (...)" or "No Warranty" is regexed for `<number> <day/week/month/year>`; unparseable/absent → both left blank |
| Featured | `"No"` | default |
| **Specifications** *(extra column — not part of Startech's own template)* | the full scraped spec table, flattened | one heading per section, indented `key: value` lines, wrapped text — the complete spec data, since the Description above only surfaces a handful of highlights |
| **Image 1, Image 2, … Image N** *(extra columns — not part of Startech's own template)* | every URL in `gallery_images` (its first shot is always the same photo as the listing thumbnail, just larger, so nothing is lost by using it), falling back to just the listing `image` if a product's gallery came back empty | each downloaded and **embedded as an actual picture**, not a URL — see the resolution note below. `N` is however many photos the largest gallery in *that file's* batch has, capped at `MAX_PRODUCT_IMAGES` (8) — most rows have fewer, and their unused Image columns for that row are just blank |

> **Description is now generated, not copied.** `build_description()` (see the
> helper-functions cell above) assembles fresh copy out of the sanitized
> `key_features` bullets and the spec table — an intro line, a features list, a
> "Specifications at a glance" excerpt, and a closing line — all branded to
> `STORE_NAME` ("Gadget and Revive") instead of Startech. Every scraped snippet
> that feeds into it (and into Short Description / Specifications) is run
> through `sanitize_text()`, which strips out any "Startech" mentions or
> startech.com.bd links first. It's still a **template**, not bespoke writing —
> read it over before anything goes live, the same as you would any generated
> copy.

> **Image resolution**: the scraped `gallery_images` URLs point at Startech's
> resize *cache* (`.../image/cache/catalog/...-500x500.webp`). `_original_image_url()`
> rewrites that to the **uncached original** at the same path (no `cache/`
> segment, no size suffix) — spot-checked across real products, that's usually
> meaningfully bigger (commonly 1000×1000 vs. the cache's 500×500, i.e. 4x the
> pixel data), and it's public with no auth needed. `_fetch_thumbnail()` tries
> that original first and only falls back to the cache URL if the original
> isn't reachable. Nothing is downscaled — the *on-screen* size is constrained
> separately, to `IMAGE_DISPLAY_SIZE` (200×200px), via `xl_img.width`/`height`
> set independently of the embedded pixel data, so the sheet's columns stay a
> sane width while the picture itself stays full resolution if you zoom in or
> print. If a photo still looks soft after this, that resolution is the
> genuine ceiling — it's whatever Startech itself has on file for that photo.

> ⚠️ **Specifications/Image columns caution**: Startech's real bulk-import
> format has neither of these — no photo column at all (its own
> `Instructions` sheet says product photos can't be attached through Excel,
> full stop) and no raw-spec-dump column either. Both are appended here **for
> your review only** (the header check inside `write_bulk_workbook()` therefore
> validates just the original `PRODUCTS_HEADER` columns, not these); if you feed
> one of these files back into Startech's actual bulk uploader, either their
> importer will ignore the unrecognized trailing columns or it may reject the
> file outright — delete them first if in doubt.

> Per the template's own instructions: products created from this file
> land as **Inactive** and tagged "Bulk Upload" — nothing goes live until
> someone opens each one, confirms/uploads the real photo on the site
> itself, reviews the generated copy, and flips it Active.

In [9]:
import openpyxl

# Self-locate the scraped JSON (and the .xlsx template, if present) instead of trusting
# OUT_DIR blindly: this notebook has been run from more than one working directory (opened
# directly vs. opened from a parent folder), so search a few likely spots rather than assume one.
def _find_file(filename, candidates, required=True):
    for c in candidates:
        p = Path(c) / filename
        if p.exists():
            return p.resolve()
    if not required:
        return None
    searched = "\n  ".join(str(Path(c) / filename) for c in candidates)
    raise FileNotFoundError(f"Could not find {filename!r}. Looked in:\n  {searched}")

_search_dirs = [OUT_DIR, Path("."), Path("startech_scrape"), Path.cwd()]

# Not required -- only the (currently commented-out) write_bulk_workbook cell further down
# reads this. Part 4 (send_to_admin) never touches the .xlsx template at all, so a missing
# sample_template.xlsx here is fine as long as you're only using the API path.
TEMPLATE_PATH = _find_file("sample_template.xlsx", _search_dirs, required=False)
BULK_OUT_PATH = TEMPLATE_PATH.parent / f"startech_{CATEGORY_NAME}_bulk_upload.xlsx" if TEMPLATE_PATH else None
TEMPLATE_CATEGORY = "Processor"   # the raw category text -- sent as-is to the admin's Part 4 ingest
                                  # endpoint (category_raw) even without the .xlsx template; matches
                                  # a row in the template's Categories sheet too, if you use that path
INCLUDE_SKU = True               # False -> leave SKU blank, let the store auto-generate
SKU_PREFIX = "GR"                # SKU = "<SKU_PREFIX>-<8-char code>", e.g. "GR-A3F9B21C" -- see generate_sku()
STORE_NAME = "Gadget and Revive" # used in the generated Description copy below; every scraped
                                  # snippet also gets Startech's own name/links stripped out of it
SPLIT_BY_BRAND = True            # only relevant to the (commented-out) .xlsx export path -- ignored by Part 4

_full_json_path = full_json_path if 'full_json_path' in dir() else _find_file(f"startech_{CATEGORY_NAME}_full.json", _search_dirs)
with open(_full_json_path, encoding="utf-8") as f:
    products = json.load(f)
print(f"Template found at: {TEMPLATE_PATH}" if TEMPLATE_PATH else "No sample_template.xlsx found -- fine for Part 4 (API), needed only for the commented-out .xlsx export.")
print(f"Loaded {len(products)} scraped products from: {_full_json_path}")

# Re-apply BRAND_FILTER/PRODUCT_LIMIT here too: if Parts 1-2 already ran this session the
# listing was pre-filtered, so this is a no-op; if Part 3 is run standalone against a
# previously-saved (unfiltered) full.json, this lets you change BRAND_FILTER/PRODUCT_LIMIT
# and re-run just Part 3 to re-slice it, without re-scraping.
products = filter_products(products)
if BRAND_FILTER or PRODUCT_LIMIT:
    print(f"Filtered to {len(products)} product(s) -- BRAND_FILTER={BRAND_FILTER!r}, PRODUCT_LIMIT={PRODUCT_LIMIT!r}.")

No sample_template.xlsx found -- fine for Part 4 (API), needed only for the commented-out .xlsx export.
Loaded 121 scraped products from: startech_processor_full.json


In [10]:
WARRANTY_RE = re.compile(r"(\d+)\s*-?\s*(day|week|month|year)s?", re.IGNORECASE)

def parse_warranty(text):
    """'03 years of Service (...)' -> (3, 'year'); 'No Warranty' -> (None, None)."""
    if not text:
        return None, None
    m = WARRANTY_RE.search(text)
    if not m:
        return None, None
    return int(m.group(1)), m.group(2).lower()


_SIZE_UNIT_RE = re.compile(r'^\d+(\.\d+)?(hz|ms|inch|"|k|w)$', re.IGNORECASE)

def extract_model(name, brand):
    """Best-effort model-number guess: first alphanumeric token (after the
    brand prefix) that contains a digit and isn't a size/spec token like
    '144Hz' or '27"'. Heuristic - spot check before trusting it."""
    if not name:
        return None
    text = name
    if brand and text.lower().startswith(brand.lower()):
        text = text[len(brand):].strip()
    for tok in text.split():
        clean = tok.strip('()",.')
        if re.search(r"[A-Za-z]", clean) and re.search(r"\d", clean) and len(clean) >= 3:
            if _SIZE_UNIT_RE.fullmatch(clean):
                continue
            return clean
    return None


def map_stock(status_text):
    s = (status_text or "").lower()
    if "in stock" in s:
        return "Yes", None
    if "out of stock" in s:
        return "No", 0
    return "No", None   # Pre Order / Up Coming / unknown


def map_price(p):
    regular = p.get("detail_price_old") or p.get("regular_price") or p.get("detail_price") or p.get("price")
    sale = p.get("detail_price") or p.get("price")
    if regular and sale and sale < regular:
        return regular, sale
    return (sale or regular), None


_STARTECH_RE = re.compile(r"https?://(www\.)?startech\.com\.bd\S*|start\s*-?\s*tech(\.com\.bd)?", re.IGNORECASE)

def sanitize_text(text):
    """Strip Startech's own name/links out of any scraped snippet, so nothing we
    build from it (Description, Short Description, Specifications) reads like
    their listing. Applied to every piece of scraped copy before it's reused."""
    if not text:
        return text
    text = _STARTECH_RE.sub("", text)
    return re.sub(r"[ \t]{2,}", " ", text).strip(" -,.")


def format_specification(specification):
    """Flatten the scraped spec table into readable multi-line text for the
    Specifications column: one heading per section, indented 'key: value' lines,
    empty sections/values dropped."""
    if not specification:
        return None
    lines = []
    for category, kvs in specification.items():
        clean_kvs = [(k, sanitize_text(v)) for k, v in kvs.items()]
        clean_kvs = [(k, v) for k, v in clean_kvs if v]
        if not clean_kvs:
            continue
        lines.append(f"{category}:")
        lines.extend(f"  • {k}: {v}" for k, v in clean_kvs)
    return "\n".join(lines) if lines else None


def build_description(name, key_features, specification):
    """Fresh, templated marketing copy assembled from the scraped facts --
    deliberately NOT Startech's own listing text (see the Part 3 caution:
    republishing their copy verbatim is a duplicate-content/copyright risk).
    Deterministic and fully offline: no external API calls, so it reads more
    like a structured product sheet than hand-written prose -- still worth a
    human pass before anything goes live, same as any templated copy."""
    parts = [f"Meet the {name or 'this product'} — now available at {STORE_NAME}."]

    clean_features = [f for f in (sanitize_text(f) for f in (key_features or [])) if f]
    if clean_features:
        parts.append("Here's what stands out:\n" + "\n".join(f"• {f}" for f in clean_features[:8]))

    highlights = []
    for kvs in (specification or {}).values():
        for k, v in kvs.items():
            v_clean = sanitize_text(v)
            if v_clean:
                highlights.append(f"{k}: {v_clean}")
    if highlights:
        parts.append("Specifications at a glance:\n" + "\n".join(f"- {line}" for line in highlights[:10]))

    if len(parts) == 1:   # no features, no specs -- generic fallback so the cell isn't just the intro line
        parts.append(f"Reach out to {STORE_NAME} for full specifications and details.")

    parts.append(f"Backed by {STORE_NAME} — grab yours today.")
    return "\n\n".join(parts)


def generate_sku(seed):
    """'<SKU_PREFIX>-<8-char code>', e.g. 'GR-A3F9B21C'. The code is a hash of
    `seed` (Startech's product_id/sku/name -- whichever is available), not
    literally random: same product -> same code every run. True per-run
    randomness would mean re-running this notebook mints a new SKU for the
    same product each time, and most stores treat SKU as the unique key on
    import -- that would create duplicate products instead of updating the
    existing one. This still doesn't expose Startech's own product_id."""
    digest = hashlib.sha1(str(seed).encode("utf-8")).hexdigest().upper()
    return f"{SKU_PREFIX}-{digest[:8]}"

In [11]:
PRODUCTS_HEADER = ["Name", "Name Bengali", "SKU", "Category", "Brand", "Price", "Discount Price",
                    "Stock Quantity", "Always In Stock", "Low Stock Threshold", "Current Cost", "Unit",
                    "Short Description", "Description", "Model", "Warranty Value", "Warranty Unit", "Featured"]

rows = []
warnings = []

for p in products:
    name = p.get("detail_name") or p.get("name")
    brand = p.get("detail_brand") or p.get("brand")
    price, discount_price = map_price(p)
    always_in_stock, stock_qty = map_stock(p.get("status") or p.get("stock_status"))

    specification = p.get("specification") or {}
    key_features = [f for f in (sanitize_text(f) for f in (p.get("key_features") or p.get("specs") or [])) if f]
    short_desc = "; ".join(key_features[:3])[:250]
    description = build_description(name, key_features, specification)

    warranty_text = specification.get("Warranty Information", {}).get("Warranty")
    warranty_value, warranty_unit = parse_warranty(warranty_text)

    model = extract_model(name, brand)

    sku = generate_sku(p.get("product_id") or p.get("sku") or name) if INCLUDE_SKU else None

    if not name:
        warnings.append(f"skipped record with no name: {p.get('url')}")
        continue
    if price is None:
        warnings.append(f"no price for {name!r} ({p.get('url')}) — row still included, Price left blank")

    # All available product photos: Part 2's scraped gallery (its first entry is always
    # the same shot as the listing thumbnail, just at a larger cached size, so nothing is
    # lost by preferring it) -- or just the listing thumbnail if the gallery came back empty.
    gallery = p.get("gallery_images") or []
    image_urls = list(dict.fromkeys(gallery)) if gallery else ([p["image"]] if p.get("image") else [])

    rows.append({
        "Name": name, "Name Bengali": None, "SKU": sku, "Category": TEMPLATE_CATEGORY, "Brand": brand,
        "Price": price, "Discount Price": discount_price, "Stock Quantity": stock_qty,
        "Always In Stock": always_in_stock, "Low Stock Threshold": None, "Current Cost": None,
        "Unit": "piece", "Short Description": short_desc, "Description": description,
        "Model": model, "Warranty Value": warranty_value, "Warranty Unit": warranty_unit, "Featured": "No",
        # not template columns -- consumed by the next cell to fill the Specifications/Image N columns
        "_specifications": format_specification(specification),
        "_image_urls": image_urls,
        # only consumed by Part 4 (send_to_admin) below -- lets the admin's review screen link
        # back to the original listing even though this dict otherwise mirrors the template's columns
        "_source_url": p.get("detail_url") or p.get("url"),
        "_source_product_id": p.get("product_id"),
    })

print(f"Mapped {len(rows)}/{len(products)} products.")
if warnings:
    print(f"{len(warnings)} warning(s), e.g.:")
    for w in warnings[:10]:
        print(" -", w)

Mapped 121/121 products.
42 warning(s), e.g.:
 - no price for 'AMD Ryzen 9 5900X Processor' (https://www.startech.com.bd/amd-ryzen-9-5900x-processor) — row still included, Price left blank
 - no price for 'AMD Ryzen 7 PRO 5750G Processor with Radeon Graphics' (https://www.startech.com.bd/amd-ryzen-7-pro-5750g-processor) — row still included, Price left blank
 - no price for 'Intel Core i3 14100F 14th Gen Raptor Lake Processor' (https://www.startech.com.bd/intel-core-i3-14100f-14th-gen-processor) — row still included, Price left blank
 - no price for 'Intel Core i7 14700F 14th Gen Raptor Lake Processor' (https://www.startech.com.bd/intel-core-i7-14700f-14th-gen-processor) — row still included, Price left blank
 - no price for 'Intel Core i9 14900F 14th Gen Raptor Lake Processor' (https://www.startech.com.bd/intel-core-i9-14900f-14th-gen-processor) — row still included, Price left blank
 - no price for 'Intel Core Ultra 5 245KF Arrow Lake Processor' (https://www.startech.com.bd/intel-cor

In [12]:
# --- Excel export commented out for now -- using Part 4's direct-to-admin-database path instead. ---
# Uncomment this whole cell (and the Validation cell right after it) if you ever want the .xlsx
# file again alongside/instead of Part 4.
# import io
# import tempfile

# from PIL import Image as PILImage
# from openpyxl.drawing.image import Image as XLImage
# from openpyxl.styles import Alignment
# from openpyxl.utils import get_column_letter

# MAX_PRODUCT_IMAGES = 8       # safety cap on embedded images per product, whatever the site returns
# IMAGE_DISPLAY_SIZE = (200, 200) # px -- how big the picture renders on screen/in the cell.
#                                # The embedded pixel data itself is NOT resized -- whatever resolution
#                                # Startech actually serves is what gets embedded verbatim (just
#                                # re-encoded to JPEG, since Excel can't embed the source .webp format).
#                                # Only the on-screen size is constrained, via xl_img.width/height below,
#                                # so columns/rows stay a sane size -- the underlying picture stays full
#                                # resolution if you zoom in or print.
# THUMB_QUALITY = 90             # JPEG quality -- kept high since we're not downscaling dimensions
# IMAGE_WORKERS = 12            # concurrent threads for thumbnail downloads -- one product can have several now
# IMAGE_ROW_HEIGHT = 150        # pt, sized to fit IMAGE_DISPLAY_SIZE's height (~1.33 px per pt)

# SPEC_COL = len(PRODUCTS_HEADER) + 1   # appended after Startech's own columns -- review-only, see Part 3 intro
# SPEC_COL_LETTER = get_column_letter(SPEC_COL)
# IMAGE_COL_START = SPEC_COL + 1


# def slugify(text):
#     """Turn a brand (or any name) into a safe folder/file-name fragment: 'PC Power' -> 'pc-power'."""
#     text = re.sub(r"[^a-z0-9]+", "-", (text or "unknown").strip().lower())
#     return text.strip("-") or "unknown"


# def _original_image_url(url):
#     """Startech serves photos through an OpenCart-style resize cache:
#     .../image/cache/catalog/.../file-500x500.webp. The *uncached* original at
#     the same path -- no "cache/" segment, no "-WxH" size suffix -- is public
#     with no auth and is usually meaningfully higher resolution (spot-checked
#     across real products: commonly 1000x1000 vs. the cache's 500x500, i.e. 4x
#     the pixel data). Returns None if `url` doesn't look like a cache path."""
#     if not url or "/image/cache/" not in url:
#         return None
#     orig = url.replace("/image/cache/", "/image/")
#     orig = re.sub(r"-\d+x\d+(?=\.\w+$)", "", orig)
#     return orig if orig != url else None


# def _fetch_thumbnail(url, dest_dir, name):
#     """Download the highest-resolution version of `url` available and save it
#     as a JPEG under dest_dir, full resolution, unresized (only the format
#     changes -- Excel can't embed .webp). Tries the uncached original first
#     (see _original_image_url), falling back to `url` itself -- the cache
#     version we already know is reachable -- if that's unavailable or fails.
#     Returns the saved path, or None if nothing works. Reuses Part 2's
#     thread-local sessions (connection reuse) and retries like fetch_one does
#     -- a single product can now need several image requests, so the same
#     transient-failure tolerance matters more here than it used to."""
#     if not url:
#         return None
#     session = get_session()
#     candidates = [u for u in (_original_image_url(url), url) if u]
#     for candidate in dict.fromkeys(candidates):  # dedup, keeps the original-first order
#         for attempt in range(1, MAX_RETRIES + 1):
#             try:
#                 resp = session.get(candidate, timeout=REQUEST_TIMEOUT)
#                 resp.raise_for_status()
#                 img = PILImage.open(io.BytesIO(resp.content)).convert("RGB")
#                 path = Path(dest_dir) / f"{name}.jpg"
#                 img.save(path, format="JPEG", quality=THUMB_QUALITY)
#                 return path
#             except Exception:
#                 if attempt == MAX_RETRIES:
#                     break   # give up on this candidate URL, try the next one (if any)
#                 time.sleep(0.5 * attempt)
#     return None


# def write_bulk_workbook(rows_subset, out_path):
#     """Build one bulk-upload workbook (Products sheet + Specifications/Image N
#     columns) from `rows_subset` and save it to `out_path`. Everything Part 3
#     used to do inline lives here now, factored out so it can run once per
#     brand (SPLIT_BY_BRAND=True) or once total (False). Returns
#     (row_count, images_embedded, images_failed)."""
#     out_path.parent.mkdir(parents=True, exist_ok=True)

#     num_image_cols = min(MAX_PRODUCT_IMAGES, max((len(r.get("_image_urls") or []) for r in rows_subset), default=0)) or 1
#     image_col_letters = [get_column_letter(IMAGE_COL_START + i) for i in range(num_image_cols)]

#     wb = openpyxl.load_workbook(TEMPLATE_PATH)
#     ws = wb["Products"]

#     # Confirm the template header hasn't drifted from what this mapping expects
#     # (checked against just Startech's own columns -- Specifications/Image N below are our own addition)
#     actual_header = [c.value for c in next(ws.iter_rows(min_row=1, max_row=1))][:len(PRODUCTS_HEADER)]
#     assert actual_header == PRODUCTS_HEADER, (
#         f"Template header changed!\nExpected: {PRODUCTS_HEADER}\nFound:    {actual_header}"
#     )

#     ws.cell(row=1, column=SPEC_COL, value="Specifications")
#     ws.column_dimensions[SPEC_COL_LETTER].width = 60
#     for i, col_letter in enumerate(image_col_letters, start=1):
#         ws.cell(row=1, column=IMAGE_COL_START + i - 1, value=f"Image {i}")
#         ws.column_dimensions[col_letter].width = 28

#     # Drop the template's sample row(s) below the header, then write our data
#     if ws.max_row > 1:
#         ws.delete_rows(2, ws.max_row - 1)

#     for row in rows_subset:
#         ws.append([row[col] for col in PRODUCTS_HEADER])

#     # Fill in the Specifications column -- wrapped, top-aligned multi-line text
#     for row_idx, row in enumerate(rows_subset, start=2):   # row 1 is the header
#         cell = ws.cell(row=row_idx, column=SPEC_COL, value=row.get("_specifications"))
#         cell.alignment = Alignment(wrap_text=True, vertical="top")

#     image_total = image_misses = 0
#     # tmpdir has to stay alive through wb.save() below -- openpyxl reads each embedded
#     # image's bytes off disk lazily at save time, not when add_image() is called.
#     with tempfile.TemporaryDirectory() as tmpdir:
#         with ThreadPoolExecutor(max_workers=IMAGE_WORKERS) as executor:
#             future_to_cell = {}
#             for row_idx, row in enumerate(rows_subset, start=2):
#                 urls = (row.get("_image_urls") or [])[:num_image_cols]
#                 for img_idx, url in enumerate(urls, start=1):
#                     image_total += 1
#                     fut = executor.submit(_fetch_thumbnail, url, tmpdir, f"{row_idx}_{img_idx}")
#                     future_to_cell[fut] = f"{image_col_letters[img_idx - 1]}{row_idx}"

#             thumb_paths = {}
#             for future in as_completed(future_to_cell):
#                 cell_ref = future_to_cell[future]
#                 path = future.result()
#                 if path is None:
#                     image_misses += 1
#                 else:
#                     thumb_paths[cell_ref] = path

#         rows_with_images = {int(re.match(r"[A-Z]+(\d+)", ref).group(1)) for ref in thumb_paths}

#         for cell_ref, path in thumb_paths.items():
#             xl_img = XLImage(str(path))
#             xl_img.width, xl_img.height = IMAGE_DISPLAY_SIZE  # embedded pixels stay full source resolution; only the on-screen size shrinks
#             ws.add_image(xl_img, cell_ref)

#         # Row height needs to fit whichever is taller for that row: the wrapped
#         # Specifications text, or an embedded image.
#         for row_idx, row in enumerate(rows_subset, start=2):
#             spec_text = row.get("_specifications")
#             n_lines = (spec_text.count("\n") + 1) if spec_text else 1
#             spec_height = min(400, max(20, n_lines * 14 + 10))
#             needed = max(spec_height, IMAGE_ROW_HEIGHT if row_idx in rows_with_images else 0)
#             ws.row_dimensions[row_idx].height = needed

#         wb.save(out_path)

#     return len(rows_subset), image_total, image_misses


# if SPLIT_BY_BRAND:
#     groups = {}
#     for row in rows:
#         groups.setdefault(row.get("Brand") or "Unknown", []).append(row)

#     written = []
#     for brand, brand_rows in sorted(groups.items()):
#         out_path = OUT_DIR / "bulk_upload" / CATEGORY_NAME / slugify(brand) / f"{CATEGORY_NAME}_{slugify(brand)}_bulk_upload.xlsx"
#         n_rows, n_total, n_miss = write_bulk_workbook(brand_rows, out_path)
#         written.append(out_path)
#         print(f"[{brand}] {n_rows} row(s) -> {out_path}  (images {n_total - n_miss}/{n_total})")

#     print(f"\nWrote {len(written)} brand workbook(s) under {OUT_DIR / 'bulk_upload' / CATEGORY_NAME}/")
# else:
#     n_rows, n_total, n_miss = write_bulk_workbook(rows, BULK_OUT_PATH)
#     print(f"Wrote {n_rows} rows -> {BULK_OUT_PATH}")
#     print(f"Embedded {n_total - n_miss}/{n_total} product images ({n_miss} failed/missing).")


### Validation

Sanity-checks the mapped output before you upload it: unknown brands
(informational only — the template auto-creates them), duplicate SKUs
within this batch, and rows missing a price.

In [13]:
# --- Commented out along with the Excel export cell above -- nothing to validate without that .xlsx. ---
# wb_check = openpyxl.load_workbook(TEMPLATE_PATH, data_only=True)
# template_brands = {r[0] for r in wb_check["Brands"].iter_rows(min_row=2, values_only=True) if r[0]}
# template_categories = {r[0] for r in wb_check["Categories"].iter_rows(min_row=2, values_only=True) if r[0]}

# assert TEMPLATE_CATEGORY in template_categories, f"{TEMPLATE_CATEGORY!r} not found in the Categories sheet!"

# unknown_brands = sorted({r["Brand"] for r in rows if r["Brand"] and r["Brand"] not in template_brands})
# print(f"Category OK: {TEMPLATE_CATEGORY!r} is a valid category.")
# print(f"{len(unknown_brands)} brand(s) not already in the template (will be auto-created on import):")
# for b in unknown_brands:
#     print("  -", b)

# skus = [r["SKU"] for r in rows if r["SKU"]]
# dupes = {s for s in skus if skus.count(s) > 1}
# print(f"\nDuplicate SKUs within this batch: {len(dupes)}" + (f" -> {sorted(dupes)}" if dupes else ""))

# no_price = [r["Name"] for r in rows if r["Price"] is None]
# print(f"Rows with no Price at all: {len(no_price)}")
# for n in no_price[:10]:
#     print("  -", n)

# no_model = sum(1 for r in rows if not r["Model"])
# no_warranty = sum(1 for r in rows if not r["Warranty Value"])
# print(f"\nRows with no Model extracted: {no_model}/{len(rows)}")
# print(f"Rows with no parseable Warranty: {no_warranty}/{len(rows)}")


## Part 4 — Send straight to the admin's staging table (alternative to Part 3)

Part 3 above writes an `.xlsx` file you then upload through the admin's Bulk Import page. This
does the same mapping (`rows`, built two cells into Part 3) but skips the spreadsheet entirely:
each product is POSTed straight into the admin's **Scraped Products** staging table
(`/scraped-products` in the admin panel), where you review it, map the category, and Promote it
into a real product yourself — same human review step either way, just without a file in between.

Needs `rows` to already exist (run Part 3's mapping cell first — you don't need to actually run
`write_bulk_workbook` / the Validation cell below it if you're only using this path).

**Fill in the config cell right below before running anything else in this section.**


In [14]:
# ============================================================
# Part 4 config -- fill these in before running the cells below.
# ============================================================
API_BASE_URL = "https://api.gadgetandrevive.com/api"  # or e.g. "http://127.0.0.1:8000/api" to try locally first
# API_TOKEN = "505|WV3fVANOTiE2MANHSRzwuRXHVXe8LjgS9nD7TTciaed48cbe"
API_TOKEN = "505|WV3fVANOTiE2MANHSRzwuRXHVXe8LjgS9nD7TTciaed48cbe"
# Generate this once, on the server, via `php artisan tinker`:
#   User::where('email', 'you@example.com')->first()->createToken('scraper-ingest')->plainTextToken
# Treat it like a password -- don't commit a real token if this notebook file ever goes into
# version control. Revoke it later the same way:
#   User::where('email', 'you@example.com')->first()->tokens()->where('name', 'scraper-ingest')->delete()

INGEST_WORKERS = 6     # concurrent submissions -- this hits our own server, not Startech, but still
                       # no reason to hammer it; lower than IMAGE_WORKERS above on purpose
MAX_INGEST_IMAGES = 6  # main image + up to this many gallery images per product

BATCH_NAME = f"{CATEGORY_NAME}-{time.strftime('%Y%m%d-%H%M%S')}"
print(f"This run's batch name (groups everything in the admin's Scraped Products list): {BATCH_NAME}")


This run's batch name (groups everything in the admin's Scraped Products list): processor-20260903-153717


### Verify `TEMPLATE_CATEGORY` against your real categories

Startech's own category names ("Monitor", "Processor", ...) don't always match yours
exactly (casing, plural/singular, a different name entirely). The cell below fetches your
**actual** category list from the admin and checks `TEMPLATE_CATEGORY` (set two cells up)
against it -- correcting casing automatically on an exact match, or warning with the
closest real names if nothing matches, so you can fix the config cell before sending
anything. This doesn't change how ingest/review works (the admin's Scraped Products review
screen still lets you map/correct the category by hand either way) -- it just catches an
obvious typo/mismatch before you scrape and submit a whole category's worth of products
under the wrong name.

In [ ]:
# Verify TEMPLATE_CATEGORY against the real category list before sending anything -- corrects
# casing/pluralization mismatches so the admin's automatic category matching (a name match
# against ProductCategory at review time) is more likely to succeed with zero manual mapping.
# Never blocks the rest of the notebook -- if this lookup itself fails (offline, server down,
# etc.) it just warns and lets you proceed with TEMPLATE_CATEGORY as typed; you can always map
# the category by hand later in the admin's Scraped Products review screen regardless of what
# happens here.
import difflib


def fetch_real_categories():
    # Public endpoint (no auth needed) -- returns every active ProductCategory as a flat list.
    resp = requests.get(
        f"{API_BASE_URL}/categories/products",
        timeout=REQUEST_TIMEOUT,
    )
    resp.raise_for_status()
    body = resp.json()
    return body.get("data", body)


try:
    real_categories = fetch_real_categories()
    real_names = [c["name"] for c in real_categories if c.get("name")]
    print(f"Fetched {len(real_names)} real categories from the admin.")

    exact = next((n for n in real_names if n.lower() == TEMPLATE_CATEGORY.lower()), None)
    if exact:
        if exact != TEMPLATE_CATEGORY:
            print(f"{TEMPLATE_CATEGORY!r} matches an existing category with different casing -- using {exact!r} instead.")
            TEMPLATE_CATEGORY = exact
        else:
            print(f"{TEMPLATE_CATEGORY!r} matches an existing category exactly. Good to go.")
    else:
        suggestions = difflib.get_close_matches(TEMPLATE_CATEGORY, real_names, n=5, cutoff=0.5)
        print(f"WARNING: {TEMPLATE_CATEGORY!r} does not exactly match any existing category.")
        if suggestions:
            print("Closest existing categories:", suggestions)
            print("Fix TEMPLATE_CATEGORY two cells up to one of these and re-run this cell, or proceed anyway --")
            print("you can still map it to the right category by hand in the admin's Scraped Products review screen.")
        else:
            print("No close match found at all -- double-check spelling, or plan to map it manually at review time.")
except Exception as e:
    print(f"Could not fetch the real category list ({e}) -- proceeding with TEMPLATE_CATEGORY={TEMPLATE_CATEGORY!r} as typed.")
    print("You can still map the category by hand later in the admin's Scraped Products review screen.")


In [15]:
import io
from PIL import Image as PILImage

THUMB_QUALITY = 90  # JPEG quality when re-encoding each downloaded image before upload -- kept
                     # high since dimensions aren't being downscaled, just re-encoded to JPEG


def _original_image_url(url):
    """Startech serves photos through an OpenCart-style resize cache:
    .../image/cache/catalog/.../file-500x500.webp. The *uncached* original at
    the same path -- no "cache/" segment, no "-WxH" size suffix -- is public
    with no auth and is usually meaningfully higher resolution (spot-checked
    across real products: commonly 1000x1000 vs. the cache's 500x500, i.e. 4x
    the pixel data). Returns None if `url` doesn't look like a cache path."""
    if not url or "/image/cache/" not in url:
        return None
    orig = url.replace("/image/cache/", "/image/")
    orig = re.sub(r"-\d+x\d+(?=\.\w+$)", "", orig)
    return orig if orig != url else None


# Defined locally (rather than reused from the Excel-export cell above) so Part 4 never depends
# on that cell -- it's fully commented out right now, and even when it isn't, this shouldn't care.
def _fetch_image_bytes(url):
    """Same highest-resolution-first download _fetch_thumbnail() does above, but returns raw
    JPEG bytes instead of saving to disk -- Part 4 attaches bytes straight to the admin API,
    there's no .xlsx for openpyxl to read them back off disk from."""
    if not url:
        return None
    session = get_session()
    candidates = [u for u in (_original_image_url(url), url) if u]
    for candidate in dict.fromkeys(candidates):
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                resp = session.get(candidate, timeout=REQUEST_TIMEOUT)
                resp.raise_for_status()
                img = PILImage.open(io.BytesIO(resp.content)).convert("RGB")
                buf = io.BytesIO()
                img.save(buf, format="JPEG", quality=THUMB_QUALITY)
                return buf.getvalue()
            except Exception:
                if attempt == MAX_RETRIES:
                    break
                time.sleep(0.5 * attempt)
    return None


def _ingest_one(row, batch):
    """POSTs one `rows` entry (see Part 3's mapping cell) to /admin/scraped-products/ingest.
    Returns (name, ok, status_or_message): on success `status_or_message` is the response's
    own `status` field (created/updated/reactivated/synced_to_product/duplicate/
    skipped_no_price -- see ScrapedProductController::ingest()); on failure it's the error
    text. A 200 is just as much a success as a 201 here -- only a brand-new row comes back
    201, everything else the endpoint does (update/sync/flag-duplicate/skip) is 200."""
    urls = (row.get("_image_urls") or [])[:1 + MAX_INGEST_IMAGES]
    main_bytes = _fetch_image_bytes(urls[0]) if urls else None
    gallery_bytes = [b for b in (_fetch_image_bytes(u) for u in urls[1:]) if b]

    data = {
        "batch": batch,
        "source_site": "startech",
        "source_url": row.get("_source_url"),
        "source_product_id": row.get("_source_product_id"),
        "category_raw": row.get("Category"),
        "brand_raw": row.get("Brand"),
        "sku": row.get("SKU"),
        "name": row.get("Name"),
        "name_bn": row.get("Name Bengali"),
        "price": row.get("Price"),
        "discount_price": row.get("Discount Price"),
        "stock_qty": row.get("Stock Quantity"),
        "stock_status_raw": "In Stock" if row.get("Always In Stock") == "Yes" else "Out of Stock",
        "unit": row.get("Unit"),
        "short_description": row.get("Short Description"),
        "description": row.get("Description"),
        "specifications": row.get("_specifications"),
        "model": row.get("Model"),
        "warranty_value": row.get("Warranty Value"),
        "warranty_unit": row.get("Warranty Unit"),
    }
    # requests sends the literal string "None" for a None value unless we strip them ourselves.
    data = {k: v for k, v in data.items() if v is not None}

    files = []
    if main_bytes:
        files.append(("image", ("main.jpg", main_bytes, "image/jpeg")))
    for i, b in enumerate(gallery_bytes):
        files.append(("gallery[]", (f"gallery_{i}.jpg", b, "image/jpeg")))

    try:
        resp = requests.post(
            f"{API_BASE_URL}/admin/scraped-products/ingest",
            headers={"Authorization": f"Bearer {API_TOKEN}"},
            data=data,
            files=files or None,
            timeout=REQUEST_TIMEOUT,
        )
        if resp.status_code in (200, 201):
            try:
                status = resp.json().get("data", {}).get("status") or "created"
            except ValueError:
                status = "created"
            return row.get("Name"), True, status
        return row.get("Name"), False, f"HTTP {resp.status_code}: {resp.text[:300]}"
    except Exception as e:
        return row.get("Name"), False, str(e)


def send_to_admin(rows_subset, batch=None):
    """Submits every row in `rows_subset` (same shape as Part 3's `rows`) to the admin's
    Scraped Products staging table -- one HTTP call per product, images downloaded fresh from
    Startech and attached as real file bytes (never a URL the server would have to fetch
    itself, which would just hit the same Cloudflare block this notebook exists to route
    around). A re-submitted product that matches one already staged or promoted (by its own
    source id) updates/syncs that row in place instead of creating a second copy -- see
    ScrapedProductController::ingest(). Prints a created/updated/synced/duplicate/skipped/failed
    summary (plus one machine-readable `INGEST_BREAKDOWN::{...}` line used by run_scrape.py when
    this notebook is triggered from the admin's "Run Scraper" button instead of run by hand) and
    returns the list of failures."""
    if not API_TOKEN or API_TOKEN == "PASTE_YOUR_TOKEN_HERE":
        raise RuntimeError("Set API_TOKEN in the config cell above first -- see the comment there for how to generate one.")

    batch = batch or BATCH_NAME
    results = []
    with ThreadPoolExecutor(max_workers=INGEST_WORKERS) as executor:
        futures = [executor.submit(_ingest_one, row, batch) for row in rows_subset]
        for future in as_completed(futures):
            results.append(future.result())

    created = [r for r in results if r[1] and r[2] == "created"]
    updated = [r for r in results if r[1] and r[2] in ("updated", "reactivated")]
    synced = [r for r in results if r[1] and r[2] == "synced_to_product"]
    duplicate = [r for r in results if r[1] and r[2] == "duplicate"]
    skipped = [r for r in results if r[1] and r[2] == "skipped_no_price"]
    failed = [r for r in results if not r[1]]

    print(
        f"Sent {len(results)} product(s) to {API_BASE_URL} -- {len(created)} created, "
        f"{len(updated)} updated, {len(synced)} synced to a live product, "
        f"{len(duplicate)} duplicate, {len(skipped)} skipped, {len(failed)} failed."
    )
    print(f"Batch: {batch} -- review them in the admin panel under Scraped Products.")
    if failed:
        print(f"\n{len(failed)} failure(s):")
        for name, _ok, message in failed[:20]:
            print(f" - {name!r}: {message}")

    import json as _json
    print("INGEST_BREAKDOWN::" + _json.dumps({
        "created": len(created), "updated": len(updated), "synced": len(synced),
        "duplicate": len(duplicate), "skipped": len(skipped), "failed": len(failed),
        "batch": batch,
    }))

    return failed


In [16]:
# Run once Part 3's mapping cell (the one that builds `rows`) has run and API_TOKEN above is
# set. Safe to re-run: a re-submitted product that matches one already staged or promoted (by
# its own source id) updates/syncs that row or live product in place instead of creating a
# second copy -- see ScrapedProductController::ingest(). Only a fuzzy sku/name-only match (no
# source id) still just gets flagged as a duplicate and left untouched.
failed_ingests = send_to_admin(rows)


Sent 121 product(s) to https://api.gadgetandrevive.com/api -- 77 staged, 44 failed.
Batch: processor-20260903-153717 -- review them in the admin panel under Scraped Products.

44 failure(s):
 - 'Intel Core i3 10105 10th Gen Comet Lake Processor': HTTP 200: {"success":true,"message":"Duplicate \u2014 a product named 'Intel Core i3 10105 10th Gen Comet Lake Processor' already exists in the live catalog","data":{"id":null,"status":"duplicate","reason":"a product named 'Intel Core i3 10105 10th Gen Comet Lake Processor' already exists in the live catalog"
 - 'AMD Ryzen 9 5900X Processor': HTTP 200: {"success":true,"message":"Skipped \u2014 no price given","data":{"id":null,"status":"skipped_no_price"}}
 - 'AMD Ryzen 7 PRO 5750G Processor with Radeon Graphics': HTTP 200: {"success":true,"message":"Skipped \u2014 no price given","data":{"id":null,"status":"skipped_no_price"}}
 - 'Intel Core i7 14700F 14th Gen Raptor Lake Processor': HTTP 200: {"success":true,"message":"Skipped \u2014 no pric